# Entrenamientos con K-fold por bloques (#20)
Entrena un modelo en cada uno de los 5 folds (train + val, 148 pares) y reporta el IoU de locación como media ± desvío. Sube las curvas y el resumen de cada fold y el resultado final a `gs://vmseg-imagenes/modelos/kfold/<NOMBRE>/`.
Para GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU. Abrir Colab con una sola cuenta de Google (la de UdeSA).

Cada experimento son ~30 min (5 folds). Si Colab se corta, poner en `FOLDS` los que faltan y volver a correr la celda de entrenamiento y la de resumen: el resumen junta todos los folds desde el bucket.


In [ ]:
import os
import sys

try:
    import google.colab
    COLAB = True
except:
    COLAB = False

RAMA = "20-entrenamientos"
REPO = "/content/vacamuerta-metano-seg"

if COLAB:
    # si ya se clonó (por ejemplo, al correr la celda de nuevo), no se vuelve a clonar
    if not os.path.exists(REPO):
        !git clone -b {RAMA} https://github.com/sofia2005222/vacamuerta-metano-seg.git {REPO}
    %cd {REPO}
    !pip install -qq -e .
    # el notebook no ve el paquete recién instalado hasta reiniciar: agrego src/ a mano
    sys.path.insert(0, os.path.join(REPO, "src"))
else:
    %cd ..
    %load_ext autoreload
    %autoreload 2



In [ ]:
# bajar los pares del bucket
if COLAB:
    from google.colab import auth
    auth.authenticate_user()
    !mkdir -p data/pares
    !gsutil -m -q cp gs://vmseg-imagenes/pares/indice.csv "gs://vmseg-imagenes/pares/*_t1.tif" "gs://vmseg-imagenes/pares/*_t2.tif" "gs://vmseg-imagenes/pares/*_mask.tif" data/pares/


In [ ]:
import torch
from vmseg.config import DatasetConfig, TrainingConfig, deterministic

# --- experimento ---
NOMBRE = "1_siamesa_6b_finetuning_ce_dice"
ARQUITECTURA = "siamesa"      # "siamesa" o "fusion"
BANDAS = (0, 1, 2, 3, 4, 5)   # (2, 1, 0) para RGB
INDICES = False               # True agrega NDVI, NDBI y BSI
MODO = "finetuning"           # "finetuning", "congelado" o "cero"
PERDIDA = "ce_dice"           # "ce", "ce_dice", "ce_lovasz" o "ce_dice_lovasz"
PESOS = None                  # [1.0, 10.0] solo con PERDIDA = "ce"
FOLDS = None                  # None = los 5; por ejemplo [3, 4] para retomar

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

dataset_config = DatasetConfig(data_root="data/pares", bandas=BANDAS, indices=INDICES,
                               num_classes=2, batch_size=8, augment=True)
train_config = TrainingConfig(epochs=50, learning_rate=1e-3, device=device,
                              checkpoints_dir=f"runs/kfold/{NOMBRE}",
                              pesos_clases=PESOS, perdida=PERDIDA)

In [ ]:
import json
from vmseg.kfold import entrenar_kfold

destino = f"gs://vmseg-imagenes/modelos/kfold/{NOMBRE}/"

def subir_fold(fold, resumen):
    """Guarda el resumen del fold y lo sube con sus curvas apenas termina."""
    carpeta = f"{train_config.checkpoints_dir}/fold{fold}"
    with open(f"{carpeta}/resumen.json", "w") as f:
        json.dump(resumen, f, indent=2)
    if COLAB:
        codigo = os.system(f"gsutil cp {carpeta}/curvas.png {carpeta}/resumen.json {destino}fold{fold}/")
        print("subido" if codigo == 0 else "NO se pudo subir", f"{destino}fold{fold}/")

resultados = entrenar_kfold(dataset_config, train_config, ARQUITECTURA, MODO,
                            folds=FOLDS, al_terminar_fold=subir_fold)

In [ ]:
import glob
from vmseg.kfold import media_desvio

# trae del bucket los resúmenes de todos los folds (por si alguno se corrió en otra sesión)
if COLAB:
    os.system(f"gsutil -q cp -r {destino}fold* {train_config.checkpoints_dir}/")
resumenes = [json.load(open(r)) for r in sorted(glob.glob(f"{train_config.checkpoints_dir}/fold*/resumen.json"))]

media, desvio = media_desvio(resumenes, "iou_val")
media10, desvio10 = media_desvio(resumenes, "iou_val_ult10")
resultado = dict(nombre=NOMBRE, arquitectura=ARQUITECTURA, bandas=list(BANDAS), indices=INDICES,
                 modo=MODO, perdida=PERDIDA, pesos_clases=PESOS, epocas=train_config.epochs,
                 lr=train_config.learning_rate, folds_corridos=[r["fold"] for r in resumenes],
                 iou_val_media=media, iou_val_desvio=desvio,
                 iou_val_ult10_media=media10, iou_val_ult10_desvio=desvio10,
                 mejores_epocas=[r["mejor_epoca"] for r in resumenes], por_fold=resumenes)
print(f"\n{NOMBRE}: IoU val {media:.3f} ± {desvio:.3f} | últimas 10: {media10:.3f} ± {desvio10:.3f} | folds {resultado['folds_corridos']}")

with open(f"{train_config.checkpoints_dir}/resultado.json", "w") as f:
    json.dump(resultado, f, indent=2)
if COLAB:
    codigo = os.system(f"gsutil cp {train_config.checkpoints_dir}/resultado.json {destino}")
    print("subido" if codigo == 0 else "NO se pudo subir", destino)